In [1]:
from Environment import *
from DDPG import *
from NN_Module import *
from config_56bus import Config
import os

import torch
import matplotlib.pyplot as plt
import scienceplots
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
from pandapower.plotting.plotly import pf_res_plotly

from loguru import logger


NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_axis_title": 28,
    "font_legend": 28,
    "dpi": 300,
    "colors": ['#0173B2', '#DE8F05', '#CC78BC', '#CA9161', '#949494']
}

### a simple logger
logger.remove()
logger.add(sys.stderr, level='DEBUG')

env_seed = 1        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l 6l 11h

agent_num = 5
agent_policy_net = []
safe_agent_net = []

### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
# state, topology, senario = env.reset(seed=env_seed)             #not change
topology = torch.cuda.FloatTensor(topology).unsqueeze(0)

def moving_average(a, n=3):
    # Padding the array to maintain the length after convolution.
    pad = np.pad(a, (n//2, n-1-n//2), mode='edge')
    ret = np.convolve(pad, np.ones(n), mode='valid') / n
    return ret
# 使用双重移动平均进行更强的平滑
def double_moving_average(data, n=15):
    """先应用一次移动平均，然后对结果再应用一次移动平均"""
    # 第一次移动平均
    smoothed1 = np.convolve(data, np.ones(n)/n, mode='valid')
    # 对结果进行填充，保持长度一致
    pad_size = len(data) - len(smoothed1)
    smoothed1 = np.pad(smoothed1, (pad_size//2, pad_size - pad_size//2), mode='edge')
    # 第二次移动平均
    smoothed2 = np.convolve(smoothed1, np.ones(n)/n, mode='valid')
    # 对结果进行填充，保持长度一致
    pad_size = len(data) - len(smoothed2)
    smoothed2 = np.pad(smoothed2, (pad_size//2, pad_size - pad_size//2), mode='edge')
    return smoothed2


### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=Config.topology_hidden_dim)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=Config.hidden_dim_56bus).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    #value_net_dict = torch.load(f'check_points/value_net/2023-06-19/Step_200_Seed_12_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-08-09/Step_250_Seed_23_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-08-15/Step_900_Seed_33_a{i}.pth')
    #policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2023-09-21/Step_900_Seed_10_a{i}.pth'))
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg/policy_net_checkpoint_a{i}.pth')

    safe_agent_net[i].load_state_dict(policy_net_dict)


### test the flexible controller
episode_reward = 0
episode_control = 0
voltage = []
q = []
cost = []

last_action = np.zeros((agent_num,1))

done_record = True
state, topology, senario = env.reset(seed=env_seed)
topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
for t in range(50):
    # 在第10次调用 topology_change
    if t == 20:
        logger.info("Changing topology at step {}", t)
        new_topology = env.topology_change(seed=1)
        topology = torch.cuda.FloatTensor(new_topology).unsqueeze(0)
    # 在第20次调用 topology_reset
    if t == 10:
        logger.info("Resetting topology at step {}", t)
        new_topology = env.topology_reset()
        topology = torch.cuda.FloatTensor(new_topology).unsqueeze(0)
    
    action = []
    for i in range(agent_num):
        action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
        action_agent = action_agent.detach().cpu().numpy()[0]
        action.append(action_agent)

    if np.min(action) < -0.3 or np.max(action) > 0.3:
        logger.warning('control output saturated! min is {}, max is {}', np.min(action), np.max(action))

    action = last_action - np.asarray(action)
    last_action = np.copy(action)
    
    try:
        next_state, reward, done = env.step(action)
    except:
        logger.error(sys.exc_info())
        logger.error('power flow not converge at {}', t)
        break

    if done and done_record:
        logger.info('stable at step {}', t)
        logger.info('stable cost is {}', episode_control)
        done_record = False

    voltage.append(state)
    q.append(action)
    state = next_state
    episode_reward += reward
    cost.append(-reward)
    episode_control += LA.norm(action, 2)

voltage_RL = np.asarray(voltage)
q_RL =  np.asarray(q)
cost_RL =  np.asarray(cost)
logger.info('control cost of flexible controller is {}',episode_control)



d:\Code\Python\Flexible_Voltage_Control\.venv\Lib\site-packages\pandapower\converter\pypower\from_ppc.py:334: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  branch_lookup.loc[is_trafo, "element"] = idx_trafo
C:\Users\wdyao\AppData\Local\Temp\ipykernel_8240\229534767.py:46: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\tensor\python_tensor.cpp:80.)
  topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
2025-09-10 11:32:08.703 | DEBUG    | Environment:reset:409 - Episode start at high volatage, highest is 1.1205411249684665 pu...
2025-09-10 11:32:08.858 | WARNING  | __main__:<module>:126 - control outp

In [5]:
# plot policy
def enhanced_policy_plot(policy_net, topology):
    """
    Enhanced policy visualization with consistent styling with trajectory plot
    Plots are arranged horizontally (1×5 layout)
    
    Args:
        policy_net: The policy network models for different buses
        topology: The power system topology indicator
        
    Returns:
        fig: Plotly figure object with enhanced styling
    """
    # Create 1×5 subplot layout (horizontal arrangement)
    fig = make_subplots(
        rows=1, cols=5,
        subplot_titles=['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'],
        horizontal_spacing=0.05  # Reduce spacing between subplots
    )
    for annotation in fig.layout.annotations:
        annotation.font.size = NATURE_CONFIG["font_title"]
        annotation.font.family = "Arial"
    
    # Use consistent color scheme with trajectory plot
    method_colors = {
        'RLC-FT': '#0072B2',  # Deep blue
        'Linear': '#999999'   # Gray
    }
    
    # Line styles and widths
    line_width = {
        'RLC-FT': 3,
        'Linear': 2
    }
    
    N = 400
    x_vals = np.linspace(10, 14, N)
    
    # Adjusted safe voltage range
    safe_range = [11.55, 12.45]  # Adjusted per requirements
    
    # Store all traces to ensure correct rendering order
    all_traces = {i: {'background': [], 'lines': []} for i in range(5)}
    
    # Create subplots for each bus
    bus_indices = [0, 1, 2, 3, 4]
    for idx, i in enumerate(bus_indices):
        # In horizontal layout, all plots are in row 1, column is the index+1
        row = 1
        col = idx + 1
        
        baseline_vals = []
        policy_vals = []
        
        # Calculate output values
        for j in range(N):
            state_val = 0.80 + 0.001 * j
            base = np.maximum(state_val - 1.05, 0) - np.maximum(0.95 - state_val, 0)
            baseline_vals.append(-base)
            
            state_tensor = torch.tensor([[state_val]])
            action_tensor = policy_net[i](state_tensor, topology)
            policy_vals.append( -action_tensor[0].detach().item() )

        baseline_vals = np.array(baseline_vals)
        
        # Apply PCHIP smoothing for better curve appearance
        window_size = 20
        policy_vals_smoothed = moving_average(policy_vals, n=window_size)

        
        # Scale linear values for consistency
        baseline_vals_scaled = 5 * baseline_vals
        
        # Add safe voltage range area
        fig.add_shape(
            type="rect",
            x0=safe_range[0], 
            x1=safe_range[1], 
            y0=-10,  # Using very wide y-range
            y1=10,   # that will be auto-scaled later
            fillcolor="rgba(144, 238, 144, 0.12)",
            line=dict(width=0),
            layer="below",
            row=row, col=col
        )
        
        # Add Safe Range to legend (only in first subplot)
        if idx == 0:
            all_traces[idx]['background'].append(
                go.Scatter(
                    x=[None], y=[None],
                    mode='lines',
                    fill='toself',
                    fillcolor="rgba(144, 238, 144, 0.12)",
                    line=dict(width=0),
                    name="Safe Range",
                    showlegend=True
                )
            )
        
        # Add zero line for reference
        fig.add_hline(
            y=0,
            line=dict(color="black", width=1, dash="dot"),
            row=row, col=col
        )
        
        # Add Linear and RLC-FT curves
        all_traces[idx]['lines'].append(
            go.Scatter(
                x=x_vals,
                y=baseline_vals_scaled,
                mode='lines',
                name='Linear',
                line=dict(
                    dash='dash', 
                    color=method_colors['Linear'],
                    width=line_width['Linear']
                ),
                showlegend=(idx == 0)
            )
        )

        all_traces[idx]['lines'].append(
            go.Scatter(
                x=x_vals,
                y=policy_vals_smoothed,
                mode='lines',
                name='RLC-FT',
                line=dict(
                    color=method_colors['RLC-FT'],
                    width=line_width['RLC-FT']
                ),
                showlegend=(idx == 0)
            )
        )
        
        # Calculate y-axis range based on data
        y_min = min(min(baseline_vals_scaled), min(policy_vals_smoothed)) * 1.1
        y_max = max(max(baseline_vals_scaled), max(policy_vals_smoothed)) * 1.1
        
        # Set axis styles
        fig.update_xaxes(
            title_text="Voltage (kV)" if col == 3 else None,  # Only middle plot gets x-axis title
            title_font=dict(size=NATURE_CONFIG['font_axis_title']+6),
            tickfont=dict(size=NATURE_CONFIG['font_axis']),
            range=[10, 14],
            showgrid=True,
            gridwidth=0.5,
            gridcolor='#E5E5E5',
            zeroline=False,
            row=row, col=col
        )
        
        # Set y-axis styles
        fig.update_yaxes(
            title_text="Q (MVar)" if col == 1 else None,
            title_font=dict(size=NATURE_CONFIG['font_axis_title']+6),
            tickfont=dict(size=NATURE_CONFIG['font_axis']),
            range=[y_min, y_max],  # Dynamically scale y-axis for each subplot
            showgrid=True,
            gridwidth=0.5,
            gridcolor='#E5E5E5',
            zeroline=False,
            row=row, col=col
        )
    
    # Add all traces in the correct order
    for idx in range(5):
        row = 1  # All in row 1 for horizontal layout
        col = idx + 1
        
        # Add background elements first
        for trace in all_traces[idx]['background']:
            fig.add_trace(trace, row=row, col=col)
        
        # Add lines next
        for trace in all_traces[idx]['lines']:
            fig.add_trace(trace, row=row, col=col)

    # Overall layout settings - adjust width/height for horizontal layout
    fig.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG['font_base'], color='black'),
        width=1800,  # Wider for horizontal layout
        height=600,  # Lower height
        margin=dict(l=60, r=30, t=100, b=60),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.10,
            xanchor="center",
            x=0.5,
            bgcolor='rgba(255,255,255,0.8)',
            font=dict(size=NATURE_CONFIG['font_legend']),
        ),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    
    # Save high-resolution images
    output_path = os.path.join(Config.data_path, 'images', '56bus', 'enhanced_policy_plot.svg')
    fig.write_image(output_path, width=1800, height=600, scale=2)
    
    # # Also save PNG version
    fig.write_image(os.path.join(Config.data_path, 'images', '56bus', 'enhanced_policy_plot.png'), width=1800, height=600, scale=2)
    
    # Display the plot
    fig.show()
    
    return fig

fig = enhanced_policy_plot(agent_policy_net, topology)

In [27]:
def enhanced_policy_plot_v2(
    policy_net,
    topology,
    bus_indices=(0, 1, 2, 3, 4),
    bus_titles=('Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'),
    x_range=(10.0, 14.0),            # kV
    safe_range=(11.5, 12.5),         # kV
    N=400,
    linear_scale=1.0,                # 若需将线性基线缩放，请设为>1
    smoothing='savgol',              # 'savgol', 'moving', None
    smooth_window=25, smooth_poly=3
):
    """
    Five-panel policy visualization comparing RLC-FT vs Linear on representative buses.
    - Consistent styling across subplots
    - Light safe-range shading
    - Global unified Y-axis range and ticks for comparability
    """
    import numpy as np
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import torch

    # ---------- helpers ----------
    def moving_average(a, n=21):
        a = np.asarray(a, float)
        if n <= 1: return a
        return np.convolve(a, np.ones(n)/n, mode='same')

    def smooth_vec(y):
        y = np.asarray(y, float)
        if smoothing is None:
            return y
        if smoothing == 'moving':
            return moving_average(y, n=max(5, smooth_window//2))
        if smoothing == 'savgol':
            try:
                from scipy.signal import savgol_filter
                win = smooth_window if smooth_window % 2 == 1 else smooth_window + 1
                win = min(win, len(y) - (1 - len(y) % 2))
                win = max(win, 5) if win % 2 == 1 else max(win-1, 5)
                win = min(win, len(y) - (1 - len(y) % 2))
                if win < 5:
                    return moving_average(y, n=max(3, len(y)//10))
                poly = min(smooth_poly, max(2, (win-1)//2))
                return savgol_filter(y, window_length=win, polyorder=poly, mode='interp')
            except Exception:
                return moving_average(y, n=max(5, smooth_window//2))
        return y

    def tolist(x):
        return np.asarray(x).tolist()

    # ---------- styles ----------
    safe_fill = "rgba(0, 158, 115, 0.12)"  # 淡绿
    color_rlc = "#0072B2"                  # 蓝
    color_lin = "#B3B3B3"                  # 浅灰
    zero_line = "rgba(0,0,0,0.55)"

    # ---------- sampling ----------
    x_vals = np.linspace(float(x_range[0]), float(x_range[1]), int(N))
    # 假设 policy_net[i] 接受电压标幺值，这里沿用你之前：0.80 + 0.001*j
    v_pu = 0.80 + 0.001 * np.arange(N)     # 若需严格以 x_vals/12，请自行替换

    # ---------- first pass: compute all data to set global Y range ----------
    all_linear, all_rlc = [], []

    # 暂存每个子图数据
    per_bus_data = []

    for i in bus_indices:
        # 线性基线（与原实现保持一致）
        baseline_vals = []
        for vp in v_pu.tolist():
            base = np.maximum(vp - 1.05, 0) - np.maximum(0.95 - vp, 0)
            baseline_vals.append(-base)
        baseline_vals = np.asarray(baseline_vals, float) * float(linear_scale)

        # RLC-FT
        q_rlc = []
        with torch.no_grad():
            for vp in v_pu.tolist():
                st = torch.as_tensor([[float(vp)]], dtype=torch.float32)
                act = policy_net[i](st, topology)
                q_rlc.append(float((-act[0].detach().cpu().numpy().item())))
        q_rlc = smooth_vec(q_rlc)

        per_bus_data.append((baseline_vals, q_rlc))
        all_linear.append(baseline_vals)
        all_rlc.append(q_rlc)

    all_linear = np.concatenate(all_linear)
    all_rlc = np.concatenate(all_rlc)
    global_min = 1.05 * float(min(all_linear.min(), all_rlc.min()))
    global_max = 1.05 * float(max(all_linear.max(), all_rlc.max()))
    # 生成均匀tick（5个刻度），对齐0
    def nice_ticks(ymin, ymax, k=5):
        span = ymax - ymin
        if span <= 0: return [ymin, ymax]
        step = span / (k - 1)
        # 四舍五入到0.1
        step = max(0.1, round(step / 0.1) * 0.1)
        ticks = np.arange(np.floor(ymin / step) * step, ymax + 0.5 * step, step)
        return ticks.tolist()
    yticks = nice_ticks(global_min, global_max, k=5)

    # ---------- build figure ----------
    fig = make_subplots(
        rows=1, cols=len(bus_indices),
        subplot_titles=list(bus_titles),
        horizontal_spacing=0.05
    )
    for ann in fig.layout.annotations:
        ann.font.size = NATURE_CONFIG.get("font_title", 18)
        ann.font.family = "Arial"

    # 循环添加子图
    for idx, i in enumerate(bus_indices):
        row, col = 1, idx + 1
        baseline_vals, q_rlc = per_bus_data[idx]

        # 安全带
        fig.add_shape(
            type="rect",
            x0=float(safe_range[0]), x1=float(safe_range[1]),
            y0=float(global_min) - 1.0, y1=float(global_max) + 1.0,   # 覆盖全高
            fillcolor=safe_fill, line=dict(width=0), layer="below",
            row=row, col=col
        )
        # 图例中的“Safe Range”占位只在第一个子图添加一次
        if idx == 0:
            fig.add_trace(
                go.Scatter(
                    x=[None], y=[None],
                    mode='lines',
                    fill='toself',
                    fillcolor=safe_fill,
                    line=dict(width=0),
                    name="Safe Range",
                    showlegend=True
                ),
                row=row, col=col
            )

        # 零线
        fig.add_hline(y=0.0, line=dict(color=zero_line, width=1, dash="dot"), row=row, col=col)

        # 线性基线（轻、短虚线）
        fig.add_trace(
            go.Scatter(
                x=tolist(x_vals), y=tolist(baseline_vals),
                mode='lines',
                name='Linear',
                line=dict(color=color_lin, width=1.8, dash='dash'),
                showlegend=(idx == 0),
                hovertemplate='Voltage=%{x:.2f} kV<br>Q=%{y:.3f} MVAr<extra></extra>'
            ),
            row=row, col=col
        )

        # RLC-FT
        fig.add_trace(
            go.Scatter(
                x=tolist(x_vals), y=tolist(q_rlc),
                mode='lines',
                name='RLC-FT',
                line=dict(color=color_rlc, width=3.0),
                showlegend=(idx == 0),
                hovertemplate='Voltage=%{x:.2f} kV<br>Q=%{y:.3f} MVAr<extra></extra>'
            ),
            row=row, col=col
        )

        # 轴
        fig.update_xaxes(
            title_text="Voltage (kV)" if col == (len(bus_indices)+1)//2 else None,
            title_font=dict(size=NATURE_CONFIG.get('font_axis_title', 16)+2),
            tickfont=dict(size=NATURE_CONFIG.get('font_axis', 12)),
            range=[float(x_range[0]), float(x_range[1])],
            showgrid=True, gridwidth=0.5, gridcolor='#E5E5E5',
            zeroline=False, dtick=1.0,
            row=row, col=col
        )
        fig.update_yaxes(
            title_text="Q (MVAr)" if col == 1 else None,
            title_font=dict(size=NATURE_CONFIG.get('font_axis_title', 16)+2),
            tickfont=dict(size=NATURE_CONFIG.get('font_axis', 12)),
            range=[float(global_min), float(global_max)],
            tickmode='array', tickvals=yticks,
            showgrid=True, gridwidth=0.5, gridcolor='#E5E5E5',
            zeroline=False,
            row=row, col=col
        )

    # ---------- layout ----------
    fig.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG.get('font_base', 16), color='black')),
    fig.update_layout(
        width=NATURE_CONFIG.get('width', 1600),
        height=600,
        margin=dict(l=60, r=30, t=90, b=60),
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.10,
            xanchor="center", x=0.5,
            bgcolor='rgba(255,255,255,0.75)',
            bordercolor='lightgray', borderwidth=1,
            font=dict(size=NATURE_CONFIG.get('font_legend', 14))
        ),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    fig.show()
    return fig

# 使用示例：
fig = enhanced_policy_plot_v2(agent_policy_net, topology,
    bus_indices=(0,1,2,3,4),
    bus_titles=('Bus 18','Bus 21','Bus 30','Bus 45','Bus 53'),
    x_range=(10.0,14.0),
    safe_range=(11.5,12.5),
    N=400,
    linear_scale=1.0,
    smoothing='savgol', smooth_window=25, smooth_poly=3)

In [7]:
def improved_x_policy_plotly(policy_net):
    """
    Plot the improved single-graph comparison of policies across different topologies,
    with optimized x-axis range to highlight differences.
    """
    import plotly.graph_objects as go
    
    # Create figure with better aspect ratio
    fig = go.Figure()
    
    # Use a colorblind-friendly palette
    # colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    # colors = ['#0173B2', '#DE8F05', '#CC78BC', '#CA9161', '#949494']
    colors = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7']
    
    # Define the safe voltage range
    safe_range = [11.48, 12.52]
    
    # # Add safe range area
    # fig.add_shape(
    #     type="rect",
    #     x0=safe_range[0], y0=-1.2,
    #     x1=safe_range[1], y1=1.2,
    #     fillcolor="rgba(144, 238, 144, 0.2)",
    #     line=dict(width=0),
    #     layer="below"
    # )
    
    # Add zero line
    fig.add_shape(
        type="line",
        x0=10.5, y0=0,
        x1=13.5, y1=0,
        line=dict(color="black", width=1, dash="dash")
    )
    
    # Plot policy curves for different topologies
    N = 400
    x_vals = np.linspace(10, 14, N)
    
    for i in range(3):
        policy_vals = []
        state, topo, senario = env.reset_topo(seed=i)
        topo_tensor = torch.cuda.FloatTensor(topo).unsqueeze(0)
        
        for j in range(N):
            state_tensor = torch.tensor([[0.80 + 0.001 * j]])
            action_tensor = policy_net[2](state_tensor, topo_tensor)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy().item()))
        
        # Apply Savitzky-Golay filtering for smoother curves
        try:
            from scipy.signal import savgol_filter
            window_size = 25  # Must be odd
            poly_order = 3    # Polynomial order
            policy_vals_smoothed = savgol_filter(policy_vals, window_size, poly_order)
        except:
            # Fallback to moving average if scipy is not available
            policy_vals_smoothed = moving_average(np.array(policy_vals), n=20)
        
        # Add trace with clear labeling
        fig.add_trace(go.Scatter(
            x=x_vals, 
            y=policy_vals_smoothed,
            mode='lines',
            name=f'Topology {i}',
            line=dict(color=colors[i], width=3)
        ))

    # 修改3：优化X轴设置
    fig.update_xaxes(
        title_text="Voltage (kV)",
        title_font=dict(size=NATURE_CONFIG['font_axis_title']),
        range=[10.5, 13.5],
        showgrid=True,
        gridwidth=0.5,
        gridcolor='#E5E5E5',
        zeroline=False,
        tickfont=dict(size=NATURE_CONFIG['font_axis']),
        # 关键修改：设置刻度
        tickmode='array',
        # tickvals=[ 10.5, 11, 11.5, 12, 12.5, 13, 13.5],  # 只显示整数刻度
        # 网格设置
        dtick=0.5,  # 网格间距保持0.5
    )

    # 修改4：优化Y轴设置
    fig.update_yaxes(
        title_text="Q (MVar)",
        title_font=dict(size=NATURE_CONFIG['font_axis_title']),
        range=[-1.0, 1.0],
        showgrid=True,
        gridwidth=0.5,
        gridcolor='#E5E5E5',
        tickfont=dict(size=NATURE_CONFIG['font_axis']),
        # 关键修改：设置刻度
        tickmode='array',
        tickvals=[-0.8, -0.4, 0, 0.4, 0.8],  # 减少刻度数量
        # 网格设置
        dtick=0.25,  # 网格间距
    )

    # Update layout with improved aspect ratio and focused x-axis range
    fig.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG['font_base'], color='black'),
        width=NATURE_CONFIG['width']/2,
        height=NATURE_CONFIG['height']/2+100,
        margin=dict(l=60, r=30, t=30, b=60),
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(
            x=1.02,
            y=1,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgray',
            borderwidth=1,
            font=dict(size=NATURE_CONFIG['font_legend']),
        ),
    )
    
    # # Add "Safe Range" to the legend
    # fig.add_trace(go.Scatter(
    #     x=[None], 
    #     y=[None], 
    #     mode='lines',
    #     fill='toself',
    #     fillcolor="rgba(144, 238, 144, 0.2)",
    #     line=dict(width=0),
    #     name="Safe Range"
    # ))
    
    # Export high-resolution images
    # output_path = os.path.join(Config.data_path, 'images', '56bus', 'x_policy_plot_improved.svg')
    # fig.write_image(output_path, scale=2)
    # fig.write_image(os.path.join(Config.data_path, 'images', '56bus', 'x_policy_plot_improved.png'), scale=2)
    fig.show()
    
    return fig

fig = improved_x_policy_plotly(agent_policy_net)

In [16]:
def x_policy_minmax_bg_plus_focus(
    policy_net,
    env,
    seeds_all=tuple(range(10)),   # 用于 min–max 背景的拓扑集合
    seeds_focus=(1, 2, 4),        # 前景高亮的 3 条拓扑（手动挑选）
    x_range=(10.5, 13.5),         # kV
    N=400,
    y_range=(-1.0, 1.0),          # MVAr
    smoothing='savgol',           # 'savgol','moving',None
    smooth_window=25, smooth_poly=3,
    bg_alpha=0.16                 # 背景带透明度（0.10~0.22之间比较合适）
):
    import numpy as np
    import plotly.graph_objects as go
    import torch

    # -------- helpers --------
    def moving_average(a, n=21):
        a = np.asarray(a, float)
        if n <= 1: return a
        return np.convolve(a, np.ones(n)/n, mode='same')

    def smooth_vec(y):
        y = np.asarray(y, float)
        if smoothing is None:
            return y
        if smoothing == 'moving':
            return moving_average(y, n=max(5, smooth_window//2))
        if smoothing == 'savgol':
            try:
                from scipy.signal import savgol_filter
                win = smooth_window if smooth_window % 2 == 1 else smooth_window + 1
                win = min(win, len(y) - (1 - len(y) % 2))
                win = max(win, 5) if win % 2 == 1 else max(win-1, 5)
                win = min(win, len(y) - (1 - len(y) % 2))
                if win < 5:
                    return moving_average(y, n=max(3, len(y)//10))
                poly = min(smooth_poly, max(2, (win-1)//2))
                return savgol_filter(y, window_length=win, polyorder=poly, mode='interp')
            except Exception:
                return moving_average(y, n=max(5, smooth_window//2))
        return y

    def tolist(x):
        return np.asarray(x).tolist()

    # 样式
    colors = ['#0072B2', '#D55E00', '#009E73']  # 三条高对比
    dashes = ['solid', 'dash', 'dot']
    bg_fill = f'rgba(0,114,178,{float(bg_alpha):.3f})'  # 与前景蓝同色系的淡背景
    zero_line = 'rgba(0,0,0,0.6)'

    # 采样
    x_vals = np.linspace(float(x_range[0]), float(x_range[1]), int(N))
    v_pu   = x_vals / 12.0
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # -------- 计算 min–max 背景 --------
    curves_all = []
    with torch.no_grad():
        for s in seeds_all:
            _, topo, _ = env.reset_topo(seed=int(s))
            topo_tensor = torch.as_tensor(topo, dtype=torch.float32, device=device).unsqueeze(0)
            q = []
            for vp in v_pu.tolist():
                st = torch.as_tensor([[float(vp)]], dtype=torch.float32, device=device)
                act = policy_net[2](st, topo_tensor)
                q.append(float((-act).detach().cpu().numpy().item()))
            curves_all.append(smooth_vec(q))
    curves_all = np.asarray(curves_all, dtype=float)  # shape (S_all, N)
    y_min = curves_all.min(axis=0)
    y_max = curves_all.max(axis=0)

    # -------- 计算前景三条 --------
    focus_curves = []
    with torch.no_grad():
        for s in seeds_focus:
            _, topo, _ = env.reset_topo(seed=int(s))
            topo_tensor = torch.as_tensor(topo, dtype=torch.float32, device=device).unsqueeze(0)
            q = []
            for vp in v_pu.tolist():
                st = torch.as_tensor([[float(vp)]], dtype=torch.float32, device=device)
                act = policy_net[2](st, topo_tensor)
                q.append(float((-act).detach().cpu().numpy().item()))
            focus_curves.append((s, smooth_vec(q)))

    # -------- 绘图 --------
    fig = go.Figure()

    # 零线
    fig.add_hline(y=0.0, line=dict(color=zero_line, width=1, dash='dash'))

    # 背景 min–max 阴影带（先画 max 再用 fill='tonexty' 闭合）
    fig.add_trace(go.Scatter(
        x=tolist(x_vals), y=tolist(y_max),
        line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=tolist(x_vals), y=tolist(y_min),
        fill='tonexty', fillcolor=bg_fill,
        line=dict(color='rgba(0,0,0,0)'),
        hoverinfo='skip',
        showlegend=False  # 去掉图例
    ))

    # 前景三条
    for i, (s, yc) in enumerate(focus_curves):
        fig.add_trace(go.Scatter(
            x=tolist(x_vals), y=tolist(yc),
            mode='lines',
            name=f'Topology {i+1}',  # 按需改为 s
            line=dict(color=colors[i % len(colors)], width=3, dash=dashes[i % len(dashes)]),
            hovertemplate=f'Topology {s+1}<br>V=%{{x:.2f}} kV<br>Q=%{{y:.3f}} MVAr<extra></extra>'
        ))

    # 坐标轴
    fig.update_xaxes(
        title_text="Voltage (kV)",
        range=[float(x_range[0]), float(x_range[1])],
        showgrid=True, gridwidth=0.5, gridcolor='#E6E6E6',
        tickfont=dict(size=NATURE_CONFIG.get('font_axis', 12)-4),
        zeroline=False, dtick=1.0
    )
    fig.update_yaxes(
        title_text="Q (MVar)",
        range=[float(y_range[0]), float(y_range[1])],
        showgrid=True, gridwidth=0.5, gridcolor='#E6E6E6',
        zeroline=False,
        tickmode='array',
        tickfont=dict(size=NATURE_CONFIG.get('font_axis', 12)-4),
        tickvals=[-0.8, -0.4, 0.0, 0.4, 0.8]
    )

    # 图例放右上角（图内）
    fig.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG.get('font_base', 16) -4 , color='black'),
        width=700,
        height=500,
        margin=dict(l=70, r=30, t=40, b=60),
        plot_bgcolor='white', paper_bgcolor='white',
        legend=dict(
            x=0.98, y=0.98, xanchor='right', yanchor='top',
            bgcolor='rgba(255,255,255,0.75)', bordercolor='lightgray', borderwidth=1,
            orientation='v',
            font=dict(size=NATURE_CONFIG.get('font_legend', 14) - 4)
        )
    )

    fig.show()
    return fig

# 使用示例：
fig = x_policy_minmax_bg_plus_focus(
    agent_policy_net, env,
    seeds_all=tuple(range(10)),
    seeds_focus=(2, 3,5,),  # 你手动挑的3条
    x_range=(10, 14), N=400,
    y_range=(-0.9, 0.9),
    bg_alpha=0.14,
    smoothing='savgol', smooth_window=25
)

# Export high-resolution images
output_path = os.path.join(Config.data_path, 'images', '56bus', 'x_policy_plot_improved.svg')
fig.write_image(output_path, scale=2)
fig.write_image(os.path.join(Config.data_path, 'images', '56bus', 'x_policy_plot_improved.png'), scale=2)

In [6]:
def plot_voltage_trajectory(voltage_RL, scenario, topology_change_steps=None):
    """
    Create a professional voltage trajectory plot with topology change indicators.
    """
    import plotly.graph_objects as go
    
    # Select bus indices to display
    bus_indices = [0, 1, 2, 3, 4]
    bus_names = ['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53']
    
    # Initialize figure
    fig = go.Figure()
    
    # Colorblind-friendly palette
    #colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    colors = ['#0173B2', '#DE8F05', '#CC78BC', '#CA9161', '#949494']
    #colors = ['#0173B2', '#F0E442', '#009E73', '#E69F00', '#999999']
    #colors = ['#1B4F72', '#E67E22', '#6C3483', '#922B21', '#515A5A']
    limit_color = '#029E73'
    nominal_line = '#CC3311'
    
    # Add voltage trajectories
    steps = list(range(voltage_RL.shape[0]))
    for i, idx in enumerate(bus_indices):
        fig.add_trace(go.Scatter(
            x=steps, 
            y=(12*voltage_RL[:, idx]).tolist(),
            mode='lines',
            name=f'{bus_names[i]} (RLC-FT)',
            line=dict(color=colors[i], width=2.5)
        ))
    
    # Add safety limits based on scenario
    if scenario == 0:  # Undervoltage scenario
        y_range = [10, 12.6]
        fig.add_hline(
            y=11.4, 
            line=dict(color=limit_color, dash='dash', width=2),
            annotation=dict(
                text='Voltage Lower Limit',
                xref='paper', yref='y',
                x=1.02, y=11.4,
                showarrow=False,
                font=dict(size=18)
            )
        )
    elif scenario == 1:  # Overvoltage scenario
        y_range = [11.8, 13.5]
        fig.add_hline(
            y=12.6, 
            line=dict(color=limit_color, width=2),
            annotation=dict(
                text='Voltage Upper Limit',
                xref='paper', yref='y',
                x=1.02, y=12.6,
                showarrow=False,
                font=dict(size=18)
            )
        )
    else:  # Both limits scenario
        y_range = [10.8, 13.2]
        fig.add_hline(
            y=12.6, 
            line=dict(color=limit_color, dash='dash', width=2),
            annotation=dict(
                text='Voltage Upper Limit',
                xref='paper', yref='y',
                x=1.02, y=12.6,
                showarrow=False,
                font=dict(size=24)
            )
        )
        fig.add_hline(
            y=11.4, 
            line=dict(color=limit_color, dash='dash', width=2),
            annotation=dict(
                text='Voltage Lower Limit',
                xref='paper', yref='y',
                x=1.02, y=11.4,
                showarrow=False,
                font=dict(size=24)
            )
        )
    
    # Add target voltage line (properly labeled on the right side)
    fig.add_hline(
        y=12.0, 
        line=dict(color=nominal_line, width=1.5),
        annotation=dict(
            text='Nominal Voltage',
            xref='paper', yref='y',
            x=1.02, y=12.0,
            showarrow=False,
            font=dict(size=18)
        )
    )
    
    # Handle multiple topology change points
    if topology_change_steps is not None:
        if isinstance(topology_change_steps, (int, float)):
            topology_change_steps = [topology_change_steps]  # Convert single value to list
            
        for i, step in enumerate(topology_change_steps):
            fig.add_vline(
                x=step,
                line=dict(color='rgba(0,0,0,0.5)', dash='dot', width=2),
                annotation=dict(
                    text=f'Topology Change {i+1}' if len(topology_change_steps) > 1 else 'Topology Change',
                    xref='x', yref='paper',
                    x=step, y=1.0,
                    showarrow=False,
                    font=dict(size=20)
                )
            )
    
    # Update layout with professional styling
    fig.update_layout(
        font=dict(family='Arial', size=16, color='black'),
        width=NATURE_CONFIG['width']/2,
        height=NATURE_CONFIG['height']/2,
        margin=dict(l=60, r=100, t=30, b=60),
        xaxis_title='Iteration Steps',
        yaxis_title='Bus voltage (kV)',
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(
            range=[0, 30],  # Shortened range as requested
            showgrid=True,
            gridwidth=0.5,
            gridcolor='#E5E5E5',
            zeroline=False,
            tickfont=dict(size=NATURE_CONFIG['font_axis']-6),
            title_font=dict(size=NATURE_CONFIG['font_axis_title']-4),
        ),
        yaxis=dict(
            range=y_range,
            showgrid=True,
            gridwidth=0.5,
            gridcolor='#E5E5E5',
            zeroline=False,
            tickfont=dict(size=NATURE_CONFIG['font_axis']-8),
            title_font=dict(size=NATURE_CONFIG['font_axis_title']-4),
            dtick=0.4,  # Set y-axis ticks to every 0.5
        ),
        legend=dict(
            x=1.03,
            y=1,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgray',
            borderwidth=1,
            font=dict(size=NATURE_CONFIG['font_legend']-8),
        ),
    )
    
    # Export high-resolution images
    import plotly.io as pio
    pio.kaleido.scope.mathjax = None
    output_path = os.path.join(Config.data_path, 'images', '56bus', 'voltage_plot.svg')
    fig.write_image(output_path, scale=2)
    fig.write_image(os.path.join(Config.data_path, 'images', '56bus', 'voltage_plot.png'), scale=2)
    fig.show()
    
    return fig

# Example usage:
state, topology, scenario = env.reset(seed=env_seed)
fig = plot_voltage_trajectory(voltage_RL, scenario=1, topology_change_steps=[10, 20])

2025-09-10 11:44:16.877 | DEBUG    | Environment:reset:409 - Episode start at high volatage, highest is 1.1205411249684665 pu...


ValueError: 
    Invalid value of type 'builtins.str' received for the 'xref' property of layout.annotation
        Received value: 'paper domain'

    The 'xref' property is an enumeration that may be specified as:
      - One of the following enumeration values:
            ['paper']
      - A string that matches one of the following regular expressions:
            ['^x([2-9]|[1-9][0-9]+)?( domain)?$']

In [22]:
def plot_voltage_trajectory_mean(
    voltage_RL,
    scenario=1,
    topology_change_steps=None,
    bus_indices=(0, 1, 2, 3, 4),
    max_steps=30,                 # 只保留前 N 个 step（含 0）
    show_markers=True,
    marker_every=1,               # 每隔几个点加一个 marker
    show_range=False,             # 是否显示 min--max 阴影带
    event_style='line',           # 'line' 用虚线, 'band' 用灰带
    event_labels=None,            # 自定义事件标签列表；默认 ['Disconnection','Reconnection',...]
    export_path_svg=None,
    export_path_png=None
):
    """
    Plot mean bus voltage (kV) over selected buses with topology-change markers.
    voltage_RL: (T, B) per-unit, converted to kV by *12.0
    """
    import numpy as np
    import plotly.graph_objects as go
    import plotly.io as pio
    import os

    T, B = voltage_RL.shape
    last_idx = min(int(max_steps), T-1) if max_steps is not None else T-1
    steps = np.arange(last_idx + 1)

    # Data in kV
    y_sel  = 12.0 * voltage_RL[:last_idx+1, list(bus_indices)]   # (N, K)
    y_mean = np.mean(y_sel, axis=1)
    y_min  = np.min(y_sel, axis=1)
    y_max  = np.max(y_sel, axis=1)

    # Colors
    mean_color    = '#0072B2'                         # blue
    range_fill    = 'rgba(0,114,178,0.18)'            # light blue alpha
    limit_color   = '#009E73'                         # green
    nominal_color = '#666666'                         # gray
    event_line_color = 'rgba(0,0,0,0.55)'             # subtle gray for event lines

    # Axis ranges by scenario
    if scenario == 0:
        y_low, y_high = 10.0, 12.6
        upper_limit, lower_limit = None, 11.4
    elif scenario == 1:
        y_low, y_high = 11.8, 13.5
        upper_limit, lower_limit = 12.6, None
    else:
        y_low, y_high = 10.8, 13.2
        upper_limit, lower_limit = 12.6, 11.4

    # Padding based on shown segment
    y_low  = min(y_low,  float(np.min(y_min)) - 0.05)
    y_high = max(y_high, float(np.max(y_max)) + 0.05)

    fig = go.Figure()

    # Event markers (line or band)
    if topology_change_steps is not None:
        if not isinstance(topology_change_steps, (list, tuple, np.ndarray)):
            topology_change_steps = [int(topology_change_steps)]
        if event_labels is None:
            # 默认给前两个事件以更具体的名称
            default_names = ['Disconnection', 'Reconnection']
            event_labels = [default_names[i] if i < len(default_names) else f'Change {i+1}'
                            for i in range(len(topology_change_steps))]
        if event_style == 'band':
            for step, lab in zip(topology_change_steps, event_labels):
                if step < 0 or step > steps[-1]: 
                    continue
                x0 = max(step - 0.5, steps[0])
                x1 = min(step + 0.5, steps[-1])
                fig.add_vrect(x0=x0, x1=x1, fillcolor='rgba(0,0,0,0.18)', line_width=0)
                fig.add_annotation(x=step, y=y_high, xref='x', yref='y',
                                   text=lab, showarrow=False, yshift=10,
                                   font=dict(size=NATURE_CONFIG.get("font_axis", 24), color='#333'))
        else:  # 'line' style (recommended with range band)
            for step, lab in zip(topology_change_steps, event_labels):
                if step < 0 or step > steps[-1]:
                    continue
                fig.add_vline(x=step, line=dict(color=event_line_color, width=1.6, dash='dash'))
                # place label above the plotting area to avoid overlap with bands
                fig.add_annotation(x=step, xref='x', y=1.08, yref='paper',
                                   text=lab, showarrow=False,
                                   font=dict(size=NATURE_CONFIG.get("font_axis",24), color='#444'),
                                   align='center')

    # Optional min--max envelope
    if show_range:
        fig.add_trace(go.Scatter(
            x=steps, y=y_max,
            line=dict(color='rgba(0,0,0,0)'), showlegend=False, hoverinfo='skip'
        ))
        fig.add_trace(go.Scatter(
            x=steps, y=y_min,
            fill='tonexty', fillcolor=range_fill,
            line=dict(color='rgba(0,0,0,0)'), showlegend=False, hoverinfo='skip'
        ))

    # Mean trajectory (with optional markers)
    if show_markers:
        line_trace = go.Scatter(
            x=steps, y=y_mean, mode='lines',
            line=dict(color=mean_color, width=2.4),
            hoverinfo='skip', showlegend=False
        )
        marker_trace = go.Scatter(
            x=steps[::max(1, marker_every)],
            y=y_mean[::max(1, marker_every)],
            mode='markers',
            marker=dict(size=10, color=mean_color, line=dict(color='white', width=0.5)),
            name=f'Mean voltage (n={len(bus_indices)})',
            hovertemplate='Step=%{x}<br>Mean V=%{y:.3f} kV<extra></extra>'
        )
        fig.add_trace(line_trace)
        fig.add_trace(marker_trace)
    else:
        fig.add_trace(go.Scatter(
            x=steps, y=y_mean, mode='lines',
            name=f'Mean voltage (n={len(bus_indices)})',
            line=dict(color=mean_color, width=2.4),
            hovertemplate='Step=%{x}<br>Mean V=%{y:.3f} kV<extra></extra>'
        ))

    # Limits and labels (offset to avoid overlap)
    if upper_limit is not None:
        fig.add_hline(y=upper_limit, line=dict(color=limit_color, width=2))
        fig.add_annotation(
            x=0.985, xref='paper', y=upper_limit, yref='y',
            text='Voltage Upper Limit',
            showarrow=False, xanchor='left',
            yshift=15, xshift=-150,
            font=dict(size=20, color=limit_color),
            bgcolor='rgba(255,255,255,0.7)'
        )
    if lower_limit is not None:
        fig.add_hline(y=lower_limit, line=dict(color=limit_color, width=2, dash='dash'))

    fig.add_hline(y=12.0, line=dict(color=nominal_color, width=1.5))
    fig.add_annotation(
        x=0.985, xref='paper', y=12.0, yref='y',
        text='Nominal Voltage',
        showarrow=False, xanchor='left',
        yshift=-15, xshift=-150,
        font=dict(size=20, color=nominal_color),
        bgcolor='rgba(255,255,255,0.7)'
    )

    # Layout
    width  = 700
    height = 500
    fig.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG.get("font_base",28)-4, color='black'),
        width=width, height=height,
        margin=dict(l=60, r=60, t=60, b=60),  # Adjust margins to give more space
        plot_bgcolor='white', paper_bgcolor='white',
        showlegend=False,
        xaxis=dict(
            range=[steps[0] - 0.5, steps[-1] + 0.5],  # Add padding to avoid marker clipping
            title='Iteration Steps',
            showgrid=True, gridcolor='#E6E6E6', gridwidth=0.5,
            zeroline=False, 
            dtick=10,
            title_standoff=20,  # Add space between axis title and numbers
            tickfont=dict(size=NATURE_CONFIG.get("font_axis", 24) - 2),
        ),
        yaxis=dict(
            range=[y_low - 0.1, y_high + 0.1],  # Add padding to avoid marker clipping
            title='Bus voltage (kV)',
            showgrid=True, gridcolor='#E6E6E6', gridwidth=0.5,
            zeroline=False, 
            dtick=0.4 if (y_high - y_low) <= 3.0 else 0.5,
            title_standoff=20,  # Add space between axis title and numbers
            tickfont=dict(size=NATURE_CONFIG.get("font_axis", 24) - 2),
        ),
    )

    # Export images
    if export_path_svg is None:
        export_path_svg = os.path.join(Config.data_path, 'images', '56bus', 'voltage_plot_mean.svg')
    if export_path_png is None:
        export_path_png = os.path.join(Config.data_path, 'images', '56bus', 'voltage_plot_mean.png')
    fig.write_image(export_path_svg, scale=2)
    fig.write_image(export_path_png, scale=2)
    fig.show()
    return fig

# Example usage:
fig = plot_voltage_trajectory_mean(voltage_RL, scenario=1, topology_change_steps=[10, 20],
                                   show_range=True, max_steps=30, event_style='line')

In [8]:
def create_combined_nature_figure(policy_net, voltage_RL, scenario, topology=None, topology_change_steps=None):
    """
    Create three separate figures and combine them with PIL for final output
    """
    import os
    from PIL import Image
    import plotly.io as pio
    import plotly.graph_objects as go
    
    # Define paths for temporary files
    temp_dir = os.path.join(Config.data_path, "images", "56bus", "temp")
    os.makedirs(temp_dir, exist_ok=True)
    
    # Paths for individual figures
    fig1_path = os.path.join(temp_dir, "policy_plot.png")
    fig2_path = os.path.join(temp_dir, "topology_plot.png")
    fig3_path = os.path.join(temp_dir, "voltage_plot.png")
    
    # Configure plotly for high resolution exports
    pio.kaleido.scope.mathjax = None
    
    #------------------------------------------------------------------------
    # Figure 1: Policy plots (subplot A)
    #------------------------------------------------------------------------
    # Create figure with subplots for policy
    fig1 = make_subplots(
        rows=1, cols=5,
        subplot_titles=['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'],
        horizontal_spacing=0.05
    )
    
    # Common styling
    method_colors = {
        'RLC-FT': '#0072B2',  # Deep blue
        'Linear': '#999999'   # Gray
    }
    
    line_width = {
        'RLC-FT': 3,
        'Linear': 2
    }
    
    N = 400
    x_vals = np.linspace(10, 14, N)
    safe_range = [11.5, 12.5]
    
    # Add buses to each subplot
    bus_indices = [0, 1, 2, 3, 4]
    for idx, bus_idx in enumerate(bus_indices):
        col = idx + 1  # Column 1-5
        
        # Add safe voltage range area
        fig1.add_shape(
            type="rect",
            x0=safe_range[0], x1=safe_range[1],
            y0=-10, y1=10,
            fillcolor="rgba(144, 238, 144, 0.2)",
            line=dict(width=0),
            layer="below",
            row=1, col=col
        )
        
        # Add zero line
        fig1.add_shape(
            type="line",
            x0=10, x1=14,
            y0=0, y1=0,
            line=dict(color="black", width=1, dash="dot"),
            row=1, col=col
        )
        
        # Calculate policy values
        baseline_vals = []
        policy_vals = []
        
        for j in range(N):
            state_val = 0.80 + 0.001 * j
            base = np.maximum(state_val - 1.05, 0) - np.maximum(0.95 - state_val, 0)
            baseline_vals.append(-base)
            
            state_tensor = torch.tensor([[state_val]])
            action_tensor = policy_net[bus_idx](state_tensor, topology)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy()[0]))

        baseline_vals = np.array(baseline_vals)
        
        # Apply smoothing
        window_size = 20
        policy_vals_smoothed = moving_average(policy_vals, n=window_size)
        
        # Scale linear values for consistency
        baseline_vals_scaled = 5 * baseline_vals
        
        # Add Linear trace
        fig1.add_trace(
            go.Scatter(
                x=x_vals,
                y=baseline_vals_scaled,
                mode='lines',
                name='Linear',
                line=dict(
                    dash='dash', 
                    color=method_colors['Linear'],
                    width=line_width['Linear']
                ),
                showlegend=(idx == 0)
            ),
            row=1, col=col
        )

        # Add RLC-FT trace
        fig1.add_trace(
            go.Scatter(
                x=x_vals,
                y=policy_vals_smoothed,
                mode='lines',
                name='RLC-FT',
                line=dict(
                    color=method_colors['RLC-FT'],
                    width=line_width['RLC-FT']
                ),
                showlegend=(idx == 0)
            ),
            row=1, col=col
        )
        
        # Add Safe Range to legend (only for first subplot)
        if idx == 0:
            fig1.add_trace(
                go.Scatter(
                    x=[None], y=[None],
                    mode='lines',
                    fill='toself',
                    fillcolor="rgba(144, 238, 144, 0.2)",
                    line=dict(width=0),
                    name="Safe Range",
                    showlegend=True
                ),
                row=1, col=1
            )
    
    # Set axis styles
    for col in range(1, 6):
        # Update x-axes
        fig1.update_xaxes(
            range=[10, 14],
            showgrid=True,
            gridwidth=0.5,
            gridcolor='#E5E5E5',
            zeroline=False,
            tickfont=dict(size=NATURE_CONFIG["font_axis"]),
            row=1, col=col
        )
        
        # Update y-axes
        fig1.update_yaxes(
            showgrid=True,
            gridwidth=0.5,
            gridcolor='#E5E5E5',
            zeroline=False,
            tickfont=dict(size=NATURE_CONFIG["font_axis"]),
            row=1, col=col
        )
    
    # Add titles only to specific subplots
    fig1.update_xaxes(
        title_text="Voltage (kV)",
        title_font=dict(size=NATURE_CONFIG["font_axis"]),
        row=1, col=3  # Middle plot
    )
    
    fig1.update_yaxes(
        title_text="Q (MVar)",
        title_font=dict(size=NATURE_CONFIG["font_axis"]),
        row=1, col=1  # First plot
    )
    
    # Add (a) label
    fig1.add_annotation(
        text="<b>(a)</b>",
        x=-0.06, y=1.05,
        xref="paper", yref="paper",
        showarrow=False,
        font=dict(size=NATURE_CONFIG["font_title"])
    )
    
    # Configure figure 1 layout
    fig1.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG["font_base"]),
        width=NATURE_CONFIG["width"],  
        height=NATURE_CONFIG["height"] * 0.45,  # Proportional to full height
        margin=dict(l=80, r=30, t=60, b=80),
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(
            orientation="h",
            y=-0.25,
            x=0.5,
            xanchor="center",
            yanchor="top",
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgray',
            borderwidth=0.5,
            font=dict(size=NATURE_CONFIG["font_legend"]),
        )
    )
    
    # Save figure 1
    fig1.write_image(fig1_path, scale=2)
    
    #------------------------------------------------------------------------
    # Figure 2: Policy comparison across topologies (subplot B)
    #------------------------------------------------------------------------
    fig2 = go.Figure()
    
    # Use colorblind-friendly palette
    topo_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    
    # Add safe range area
    fig2.add_shape(
        type="rect",
        x0=11.48, y0=-1.2,
        x1=12.52, y1=1.2,
        fillcolor="rgba(144, 238, 144, 0.2)",
        line=dict(width=0),
        layer="below"
    )
    
    # Add zero line
    fig2.add_shape(
        type="line",
        x0=10, x1=14,
        y0=0, y1=0,
        line=dict(color="black", width=1, dash="dash")
    )
    
    # Plot policy curves for different topologies
    N = 400
    x_vals = np.linspace(10, 14, N)
    
    for i in range(5):
        policy_vals = []
        state, topo, senario = env.reset_topo(seed=i)
        topo_tensor = torch.cuda.FloatTensor(topo).unsqueeze(0)
        
        for j in range(N):
            state_tensor = torch.tensor([[0.80 + 0.001 * j]])
            action_tensor = policy_net[2](state_tensor, topo_tensor)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy()[0]))
        
        # Apply smoothing
        window_size = 25
        policy_vals_smoothed = moving_average(np.array(policy_vals), n=window_size)
        
        # Add trace
        fig2.add_trace(
            go.Scatter(
                x=x_vals, 
                y=policy_vals_smoothed,
                mode='lines',
                name=f'Topology {i}',
                line=dict(color=topo_colors[i], width=2.5)
            )
        )
    
    # Add Safe Range to legend
    fig2.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode='lines',
            fill='toself',
            fillcolor="rgba(144, 238, 144, 0.2)",
            line=dict(width=0),
            name="Safe Range"
        )
    )
    
    # Add (b) label
    fig2.add_annotation(
        text="<b>(b)</b>",
        x=-0.12, y=1.05,
        xref="paper", yref="paper",
        showarrow=False,
        font=dict(size=NATURE_CONFIG["font_title"])
    )
    
    # Configure figure 2 layout
    fig2.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG["font_base"]),
        width=NATURE_CONFIG["width"] * 0.45,  # Half the total width 
        height=NATURE_CONFIG["height"] * 0.45,  # Proportional
        margin=dict(l=80, r=30, t=60, b=80),
        xaxis=dict(
            title_text="Voltage (kV)",
            title_font=dict(size=NATURE_CONFIG["font_axis"]),
            range=[10.0, 14.0],
            showgrid=True,
            gridwidth=0.5,
            gridcolor='#E5E5E5',
            zeroline=False,
            tickfont=dict(size=NATURE_CONFIG["font_axis"]),
        ),
        yaxis=dict(
            title_text="Q (MVar)",
            title_font=dict(size=NATURE_CONFIG["font_axis"]),
            range=[-1.0, 1.0],
            showgrid=True,
            gridwidth=0.5,
            gridcolor='#E5E5E5',
            zeroline=False,
            tickfont=dict(size=NATURE_CONFIG["font_axis"]),
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(
            x=1.02, y=1.0,
            xanchor="left", yanchor="top",
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgray',
            borderwidth=0.5,
            font=dict(size=NATURE_CONFIG["font_legend"]),
        )
    )
    
    # Save figure 2
    fig2.write_image(fig2_path, scale=2)
    
    #------------------------------------------------------------------------
    # Figure 3: Voltage trajectory (subplot C)
    #------------------------------------------------------------------------
    fig3 = go.Figure()
    
    # Select bus indices to display
    bus_indices = [0, 1, 2, 3, 4]
    bus_names = ['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53']
    
    # Add voltage trajectories
    steps = list(range(voltage_RL.shape[0]))
    for i, idx in enumerate(bus_indices):
        fig3.add_trace(
            go.Scatter(
                x=steps, 
                y=(12*voltage_RL[:, idx]).tolist(),
                mode='lines',
                name=f'{bus_names[i]} (RLC-FT)',
                line=dict(color=topo_colors[i], width=2.5)
            )
        )
    
    # Add safety limits based on scenario
    if scenario == 1:  # Overvoltage scenario
        y_range = [11.8, 13.5]
        fig3.add_shape(
            type="line",
            x0=0, x1=30,
            y0=12.6, y1=12.6,
            line=dict(color="green", width=2)
        )
        fig3.add_annotation(
            text="Voltage Upper Limit",
            x=30, y=12.6,
            xref="x", yref="y",
            xanchor="left",
            showarrow=False,
            font=dict(size=NATURE_CONFIG["font_axis"]),
        )
    
    # Add target voltage line
    fig3.add_shape(
        type="line",
        x0=0, x1=30,
        y0=12.0, y1=12.0,
        line=dict(color="red", width=1.5)
    )
    fig3.add_annotation(
        text="Nominal Voltage",
        x=30, y=12.0,
        xref="x", yref="y",
        xanchor="left",
        showarrow=False,
        font=dict(size=NATURE_CONFIG["font_axis"]),
    )
    
    # Handle topology change points
    if topology_change_steps is not None:
        if isinstance(topology_change_steps, (int, float)):
            topology_change_steps = [topology_change_steps]
            
        for i, step in enumerate(topology_change_steps):
            fig3.add_shape(
                type="line",
                x0=step, x1=step,
                y0=11.8, y1=13.5,
                line=dict(color="rgba(0,0,0,0.5)", dash="dot", width=2)
            )
            fig3.add_annotation(
                text=f'Topology Change {i+1}' if len(topology_change_steps) > 1 else 'Topology Change',
                x=step, y=13.5,
                xref="x", yref="y",
                yanchor="bottom",
                showarrow=False,
                font=dict(size=NATURE_CONFIG["font_axis"]),
            )
    
    # Add (c) label
    fig3.add_annotation(
        text="<b>(c)</b>",
        x=-0.12, y=1.05,
        xref="paper", yref="paper",
        showarrow=False,
        font=dict(size=NATURE_CONFIG["font_title"])
    )
    
    # Configure figure 3 layout
    fig3.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG["font_base"]),
        width=NATURE_CONFIG["width"] * 0.45,  # Half the total width
        height=NATURE_CONFIG["height"] * 0.45,  # Proportional
        margin=dict(l=80, r=30, t=60, b=80),
        xaxis=dict(
            title_text="Iteration Steps",
            title_font=dict(size=NATURE_CONFIG["font_axis"]),
            range=[0, 30],
            showgrid=True,
            gridwidth=0.5,
            gridcolor='#E5E5E5',
            zeroline=False,
            tickfont=dict(size=NATURE_CONFIG["font_axis"]),
        ),
        yaxis=dict(
            title_text="Bus voltage (kV)",
            title_font=dict(size=NATURE_CONFIG["font_axis"]),
            range=[11.8, 13.5],
            showgrid=True,
            gridwidth=0.5,
            gridcolor='#E5E5E5',
            zeroline=False,
            tickfont=dict(size=NATURE_CONFIG["font_axis"]),
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(
            x=1.02, y=1.0,
            xanchor="left", yanchor="top",
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgray',
            borderwidth=0.5,
            font=dict(size=NATURE_CONFIG["font_legend"]),
        )
    )
    
    # Save figure 3
    fig3.write_image(fig3_path, scale=2)
    
    #------------------------------------------------------------------------
    # Combine images using PIL
    #------------------------------------------------------------------------
    try:
        # Open individual images
        img1 = Image.open(fig1_path)
        img2 = Image.open(fig2_path)
        img3 = Image.open(fig3_path)
        
        # Get dimensions
        width1, height1 = img1.size
        width2, height2 = img2.size
        width3, height3 = img3.size
        
        # Create new composite image
        composite_width = max(width1, width2 + width3)
        composite_height = height1 + max(height2, height3)
        
        composite = Image.new('RGB', (composite_width, composite_height), (255, 255, 255))
        
        # Paste images
        composite.paste(img1, (0, 0))
        composite.paste(img2, (0, height1))
        composite.paste(img3, (width2, height1))
        
        # Save output
        final_output = os.path.join(Config.data_path, "images", "56bus", "combined_nature_figure.png")
        composite.save(final_output, quality=95)
        
        # Also save PDF (note: PIL has limited PDF support, consider other libraries for this)
        pdf_output = os.path.join(Config.data_path, "images", "56bus", "combined_nature_figure.pdf")
        composite.save(pdf_output, "PDF", resolution=300.0, save_all=True)
        
        print(f"Combined figure saved to {final_output}")
        
        # Clean up temp files if desired
        # for file in [fig1_path, fig2_path, fig3_path]:
        #     os.remove(file)
            
        return composite
        
    except Exception as e:
        print(f"Error combining images: {e}")
        print("Individual figures were saved successfully, but combining failed.")
        return None

In [9]:
# Call the combined function with your data
combined_fig = create_combined_nature_figure(
    policy_net=agent_policy_net,
    voltage_RL=voltage_RL, 
    scenario=1,  # Overvoltage scenario
    topology=topology,
    topology_change_steps=[10, 20]  # Two topology change points
)

Combined figure saved to D:/Code/Python/Flexible_Voltage_Control/images\56bus\combined_nature_figure.png
